
# Notebook 03b — Metric Views: the governed semantic layer

**The single most important idea in this workshop:** a Genie agent is only as good as the data foundation under it. **Move the logic to the left** — define your KPIs and joins **once**, in a governed object in Unity Catalog, instead of hoping Genie re-derives them correctly on every question.

A **[metric view](https://learn.microsoft.com/en-us/azure/databricks/uc-semantics/metric-views/)** is that object. It pins:
- the **join paths** between fact and dimension tables,
- the **exact formula** for each KPI (OEE, first-pass yield, scrap rate, …), including the `× 100` scaling,
- the **dimensions** you're allowed to slice by.

Genie (and dashboards, and SQL users) then all read the *same* trusted definitions.

| | Tables (03, 04) | Metric view (here) |
|---|---|---|
| KPI formula | Genie must learn it (examples/measures) | **Fixed** in the view |
| Joins | Genie must pick them | **Fixed** in the view |
| Governance | per-table grants | one governed object, reusable |
| Flexibility | open-ended exploration | limited to the view's perimeter |

**Two things make a metric-view agent actually perform** (both below):
1. A **well-scoped view** — the dimensions and measures the questions need, with formulas baked in.
2. A **curated agent** — a clear instruction plus **example `MEASURE()` queries**, so Genie writes the metric-view dialect correctly.

**Perimeter trade-off (teach this):** a metric view is deterministic *within its scope* but can't answer questions outside it. This daily-quality view is perfect for KPI-by-dimension questions (OEE/yield/scrap by plant, state, product…); event-level questions (per-event defect counts, shift analysis) belong to the table-based Knowledge Store agent in 04. Pair them in production.

> **Verify the DDL in your workspace.** Metric-view YAML syntax has evolved across releases; if `CREATE VIEW ... WITH METRICS` errors, check the docs for your workspace version.

**Before you start:** run notebooks **02** (data) and **03** (baseline agent).

**Compute:** Serverless.

In [ ]:
%run ./00_workshop_config

## Connect to the workspace and pick a warehouse

Create the SDK client and select a running SQL warehouse for the agent to run its generated SQL on.

In [ ]:
from databricks.sdk import WorkspaceClient
import re
import json
import uuid
import requests

w = WorkspaceClient()
host = w.config.host.rstrip("/")
headers = {**w.config.authenticate(), "Content-Type": "application/json"}


def genie_ui_room_url(space_id: str) -> str:
    m = re.search(r"adb-(\d+)\.", host)
    o = m.group(1) if m else ""
    q = f"?o={o}" if o else ""
    return f"{host}/genie/rooms/{space_id}{q}"


def new_id():
    return uuid.uuid4().hex


warehouse_id = None
for wh in w.warehouses.list():
    if str(wh.state).upper() in ("RUNNING", "STARTING"):
        warehouse_id = wh.id
        break
if not warehouse_id:
    whs = list(w.warehouses.list())
    warehouse_id = whs[0].id if whs else None
if not warehouse_id:
    raise RuntimeError("No SQL warehouse found. Create or start one, then re-run.")
print("Warehouse:", warehouse_id)

## 1. Create the metric view

`mv_line_quality` sits on `quality_metrics_daily` (the daily aggregate) and joins to `production_lines` and `plants`. It exposes the dimensions you can slice by (including `line_status`) and the KPI measures with formulas **and scaling** baked in — `avg_oee_pct` and `avg_first_pass_yield_pct` are already 0–100, so nobody re-derives or mis-scales them.

In [ ]:
ddl = f"""
CREATE OR REPLACE VIEW {METRIC_VIEW_LINE_QUALITY}
WITH METRICS
LANGUAGE YAML
AS $$
version: 0.1
source: {fqn}.quality_metrics_daily
joins:
  - name: line
    source: {fqn}.production_lines
    on: source.production_line_id = line.line_id
  - name: plant
    source: {fqn}.plants
    on: source.plant_id = plant.plant_id
dimensions:
  - name: date
    expr: date
  - name: plant_name
    expr: plant.plant_name
  - name: state
    expr: plant.state
  - name: line_name
    expr: line.line_name
  - name: product_type
    expr: line.product_type
  - name: line_status
    expr: line.status
measures:
  - name: avg_oee_pct
    expr: AVG(oee_score) * 100
  - name: avg_first_pass_yield_pct
    expr: AVG(first_pass_yield) * 100
  - name: scrap_rate_pct
    expr: 100.0 * SUM(scrap_count) / NULLIF(SUM(units_produced), 0)
  - name: defect_rate_pct
    expr: 100.0 * SUM(defects_found) / NULLIF(SUM(units_produced), 0)
  - name: total_units_produced
    expr: SUM(units_produced)
  - name: total_defects
    expr: SUM(defects_found)
  - name: total_downtime_minutes
    expr: SUM(downtime_minutes)
  - name: avg_downtime_minutes
    expr: AVG(downtime_minutes)
  - name: distinct_plants
    expr: COUNT(DISTINCT plant_id)
  - name: distinct_lines
    expr: COUNT(DISTINCT production_line_id)
$$
"""
spark.sql(ddl)
print("Created metric view:", METRIC_VIEW_LINE_QUALITY)

## 2. Query it with `MEASURE()`

Select dimensions and wrap measures in `MEASURE(...)`. The engine applies the pinned joins and formulas — the same numbers every time, for everyone.

In [ ]:
display(spark.sql(f"""
SELECT
  state,
  ROUND(MEASURE(avg_oee_pct), 2)              AS avg_oee_pct,
  ROUND(MEASURE(avg_first_pass_yield_pct), 2) AS avg_fpy_pct,
  ROUND(MEASURE(scrap_rate_pct), 2)           AS scrap_rate_pct,
  CAST(MEASURE(total_units_produced) AS BIGINT) AS units_produced
FROM {METRIC_VIEW_LINE_QUALITY}
WHERE YEAR(date) = 2024
GROUP BY state
ORDER BY avg_oee_pct DESC
"""))

## 2b. Performance: metric view vs. raw tables

A governed semantic layer is only worth it if it doesn't cost you much at query time. A metric view compiles to essentially the **same** joins and aggregations you'd write by hand, plus a little semantic-layer planning.

The cell below (1) runs `ANALYZE ... COMPUTE STATISTICS` on the source tables, (2) times each metric-view query against its hand-written raw-table equivalent, warm, a few times, and (3) prints the medians side by side.

On this tiny workshop dataset expect the two to land **within a few hundred milliseconds** — the metric view adds a small, roughly constant planning overhead that's swamped by fixed query cost here. That constant is irrelevant in absolute terms, and it inverts at production scale, where the metric view's pinned pre-aggregations and pruning typically make it the *faster* option. The metric view's real win isn't raw speed on small data — it's pinned, governed, correct KPIs at negligible cost.

In [ ]:
import time, statistics

MV = METRIC_VIEW_LINE_QUALITY

# 1) Give the planner statistics so both paths are optimized fairly.
for _t in ["quality_metrics_daily", "production_lines", "plants"]:
    try:
        spark.sql(f"ANALYZE TABLE {fqn}.{_t} COMPUTE STATISTICS FOR ALL COLUMNS")
    except Exception as e:
        print(f"  ANALYZE {_t}: {str(e)[:80]}")

# 2) Metric-view query vs. its hand-written raw-table equivalent (matches the 08 questions).
PAIRS = {
    "Avg OEE % by state": (
        f"SELECT state, MEASURE(avg_oee_pct) FROM {MV} GROUP BY state",
        f"SELECT p.state, AVG(q.oee_score) * 100 FROM {fqn}.quality_metrics_daily q "
        f"JOIN {fqn}.plants p ON q.plant_id = p.plant_id GROUP BY p.state",
    ),
    "Scrap rate %, Texas, 2024": (
        f"SELECT MEASURE(scrap_rate_pct) FROM {MV} WHERE state='Texas' AND YEAR(date)=2024",
        f"SELECT 100.0*SUM(q.scrap_count)/NULLIF(SUM(q.units_produced),0) "
        f"FROM {fqn}.quality_metrics_daily q JOIN {fqn}.plants p ON q.plant_id=p.plant_id "
        f"WHERE p.state='Texas' AND YEAR(q.date)=2024",
    ),
    "Total units, New York, 2024": (
        f"SELECT MEASURE(total_units_produced) FROM {MV} WHERE state='New York' AND YEAR(date)=2024",
        f"SELECT SUM(q.units_produced) FROM {fqn}.quality_metrics_daily q "
        f"JOIN {fqn}.plants p ON q.plant_id=p.plant_id WHERE p.state='New York' AND YEAR(q.date)=2024",
    ),
}

def _median_secs(sql, runs=5):
    spark.sql(sql).collect()  # warm up (plan + cache)
    times = []
    for _ in range(runs):
        t0 = time.time()
        spark.sql(sql).collect()
        times.append(time.time() - t0)
    return statistics.median(times)

print(f"{'Question':<30}{'Metric view':>13}{'Raw tables':>13}{'Delta':>9}   Verdict")
print("-" * 82)
for label, (mv_sql, raw_sql) in PAIRS.items():
    mv_t = _median_secs(mv_sql)
    raw_t = _median_secs(raw_sql)
    delta = mv_t - raw_t
    # On this small dataset both are dominated by fixed planning/round-trip cost, so treat
    # anything inside a sub-second band as parity (the semantic layer's overhead is a constant).
    comparable = mv_t <= max(raw_t * 1.25, raw_t + 0.30)
    verdict = "comparable" if comparable else "slower - investigate"
    print(f"{label:<30}{mv_t:>11.2f}s{raw_t:>12.2f}s{delta:>+8.2f}s   {verdict}")
print("-" * 82)
print("On this tiny dataset the metric view adds a small (~0.1-0.2s) constant planning overhead,")
print("negligible in absolute terms. It compiles to the same plan as the raw SQL, and at")
print("production scale its pinned pre-aggregations and pruning typically make it the faster option.")

## 3. Build a *curated* Genie agent on the metric view

**Agent C.** Data source: the metric view only — so joins and KPI math are correct by construction. But we don't stop there: the agent gets a clear instruction about the `MEASURE()` dialect **and example queries** for the common patterns. That curation is what turns "has a metric view" into "answers reliably."

In [ ]:
MV = METRIC_VIEW_LINE_QUALITY

mv_instruction = (
    f"This agent answers questions from ONE governed metric view: {MV}. "
    "Query it by selecting dimensions and wrapping measures in MEASURE(<measure_name>); "
    "filter on dimensions in WHERE and GROUP BY the requested dimensions. "
    "avg_oee_pct and avg_first_pass_yield_pct are ALREADY percentages (0-100) — never multiply by 100 again. "
    "scrap_rate_pct and defect_rate_pct are percentages from the daily quality table. "
    "Dimensions: date, plant_name, state, line_name, product_type, line_status. "
    "For a calendar-year filter use YEAR(date) = <year>. "
    "For a single-value question, return just the aggregated measure (a scalar). "
    "Do NOT recompute KPIs from base tables — the metric view already encodes the joins and formulas."
)


def mv_example(question, sql):
    return {"id": new_id(), "question": [question], "sql": [sql],
            "parameters": [], "usage_guidance": []}


mv_examples = sorted([
    mv_example(
        "What is the average OEE percentage for Michigan plants in 2024?",
        f"SELECT ROUND(MEASURE(avg_oee_pct), 2) AS avg_oee_pct FROM {MV} "
        f"WHERE state = 'Michigan' AND YEAR(date) = 2024"),
    mv_example(
        "What is the scrap rate percentage for Texas plants in 2024?",
        f"SELECT ROUND(MEASURE(scrap_rate_pct), 2) AS scrap_rate_pct FROM {MV} "
        f"WHERE state = 'Texas' AND YEAR(date) = 2024"),
    mv_example(
        "What is the average first-pass yield percentage by product type in 2024?",
        f"SELECT product_type, ROUND(MEASURE(avg_first_pass_yield_pct), 2) AS fpy_pct FROM {MV} "
        f"WHERE YEAR(date) = 2024 GROUP BY product_type ORDER BY fpy_pct DESC"),
    mv_example(
        "How many total units were produced across New York plants in 2024?",
        f"SELECT CAST(MEASURE(total_units_produced) AS BIGINT) AS units FROM {MV} "
        f"WHERE state = 'New York' AND YEAR(date) = 2024"),
], key=lambda e: e["id"])


def build_serialized_metric_view():
    return json.dumps({
        "version": 2,
        "config": {"sample_questions": []},
        "data_sources": {
            "tables": [],
            "metric_views": [{
                "identifier": MV,
                "description": [
                    "Governed daily line-quality metrics. Measures (MEASURE()): avg_oee_pct, "
                    "avg_first_pass_yield_pct (both already 0-100), scrap_rate_pct, defect_rate_pct, "
                    "total_units_produced, total_downtime_minutes, avg_downtime_minutes, total_defects, "
                    "distinct_plants, distinct_lines. Dimensions: date, plant_name, state, line_name, "
                    "product_type, line_status."
                ],
            }],
        },
        "instructions": {
            "text_instructions": [{"id": new_id(), "content": [mv_instruction]}],
            "example_question_sqls": mv_examples,
        },
    })


def _list_spaces():
    r = requests.get(f"{host}/api/2.0/genie/spaces", headers=headers)
    r.raise_for_status()
    return r.json().get("spaces", [])


def create_or_update_genie_space(title, description, serialized_space_str):
    for s in _list_spaces():
        if s.get("title") == title:
            sid = s.get("space_id") or s.get("id")
            pr = requests.patch(
                f"{host}/api/2.0/genie/spaces/{sid}", headers=headers,
                json={"title": title, "description": description,
                      "warehouse_id": warehouse_id, "serialized_space": serialized_space_str},
            )
            print(f"Updated existing: {title!r} -> {sid} ({pr.status_code})")
            if pr.status_code not in (200, 201):
                print("  ", pr.text[:400])
            return sid, genie_ui_room_url(sid)
    resp = requests.post(
        f"{host}/api/2.0/genie/spaces", headers=headers,
        json={"title": title, "description": description,
              "warehouse_id": warehouse_id, "serialized_space": serialized_space_str},
    )
    if resp.status_code not in (200, 201):
        raise RuntimeError(f"Genie create failed {resp.status_code}: {resp.text[:800]}")
    sid = resp.json().get("space_id") or resp.json().get("id")
    print(f"Created: {title!r} -> {sid}")
    return sid, genie_ui_room_url(sid)


mv_id, mv_url = create_or_update_genie_space(
    GENIE_TITLE_METRIC_VIEW, GENIE_DESC_METRIC_VIEW, build_serialized_metric_view()
)

save_config_keys([
    {"key": CFG_KEY_METRIC_VIEW, "value": mv_id,
     "space_name": GENIE_TITLE_METRIC_VIEW, "space_url": mv_url},
])

print()
print("Metric-view agent:", mv_url)
print('Try: "What is the average OEE percentage for Michigan plants in 2024?"')
print("The pinned formulas + example MEASURE() queries make it deterministic.")

## Next

- **04 — Knowledge Store:** curate the *tables* with measures, filters, fields, joins, synonyms, and example SQL (the primary agent).
- **08** compares Baseline vs. **Metric View** vs. Knowledge Store on a set of KPI-by-dimension questions — both curated agents should clearly beat the baseline.

**Takeaway for your own domain:** for the KPIs your business runs on, define them once in a metric view *and* give the agent a couple of example `MEASURE()` queries. That combination is the highest-leverage thing you can do before pointing Genie at your data.